# 01 — NOAA OISST Data Exploration

อ่านข้อมูล SST รายวันจาก NOAA OISST ปี 2021 บริเวณ -10° ถึง 35°N / 100° ถึง 160°E


In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import xarray as xr
from torch.utils.data import DataLoader, Dataset

try:
    from IPython.display import display
except ImportError:  # รันนอก Jupyter
    display = print

# --- Paths ---
PROJECT_DIR = Path("/home/kankrit/Projects/ocean-forecasting")
RAW_DIR = PROJECT_DIR / "data" / "raw"
REPORT_DIR = PROJECT_DIR / "reports"
MODEL_DIR = PROJECT_DIR / "models"
for d in (RAW_DIR, REPORT_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- พื้นที่ศึกษา ---
LAT_MIN, LAT_MAX = -10.0, 35.0
LON_MIN, LON_MAX = 100.0, 160.0

# --- ช่วงเวลา ---
YEARS = range(2017, 2022)
TEST_START = pd.Timestamp("2021-01-01")  # ปี 2021 = Test set

# --- โมเดล / การเทรน ---
INPUT_DAYS = 7          # ใช้ข้อมูลย้อนหลังกี่วัน
EVAL_HORIZONS = (1, 3, 7)
BATCH_SIZE = 4
EPOCHS = 15
PATIENCE = 3
LR = 1e-3
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [2]:

# ## 2. โหลด SST (NOAA OISST)

def find_coord_name(ds, candidates):
    for name in candidates:
        if name in ds.coords or name in ds.dims or name in ds.variables:
            return name
    raise KeyError(f"Could not find any of these coordinates: {candidates}.")


def axis_slice(values, vmin, vmax):
    """slice ที่ถูกทิศ ไม่ว่าพิกัดจะเรียงจากน้อยไปมากหรือมากไปน้อย"""
    return slice(vmin, vmax) if values[0] < values[-1] else slice(vmax, vmin)


def load_oisst_year(year):
    """อ่านไฟล์ OISST ของปีนั้น แล้วตัดเฉพาะพื้นที่ศึกษา"""
    path = RAW_DIR / f"sst.day.mean.{year}.nc"
    if not path.exists():
        raise FileNotFoundError(f"❌ ไม่พบไฟล์ {path}")
    print(f"⏳ Reading local file for {year}...")
    with xr.open_dataset(path) as ds:
        lat_name = find_coord_name(ds, ("lat", "latitude"))
        lon_name = find_coord_name(ds, ("lon", "longitude"))
        time_name = find_coord_name(ds, ("time",))
        subset = ds["sst"].sel({
            time_name: slice(f"{year}-01-01", f"{year}-12-31"),
            lat_name: axis_slice(ds[lat_name].values, LAT_MIN, LAT_MAX),
            lon_name: axis_slice(ds[lon_name].values, LON_MIN, LON_MAX),
        }).load()
    return subset.to_dataset(name="sst")


def get_oisst_year(year):
    """ใช้ cache ถ้ามี ไม่งั้นอ่านจากไฟล์ต้นฉบับแล้วบันทึก cache"""
    cache = RAW_DIR / f"oisst_{year}_lat{LAT_MIN}_{LAT_MAX}_lon{LON_MIN}_{LON_MAX}.nc"
    if cache.exists():
        print(f"Using cache: {year}")
        with xr.open_dataset(cache) as ds:
            return ds.load()
    ds = load_oisst_year(year)
    ds.to_netcdf(cache)
    print(f"Saved cache: {year}")
    return ds


sst_years = [get_oisst_year(y) for y in YEARS]
time_name = find_coord_name(sst_years[0], ("time",))
ocean = xr.concat(sst_years, dim=time_name).sortby(time_name)
lat_name = find_coord_name(ocean, ("lat", "latitude"))
lon_name = find_coord_name(ocean, ("lon", "longitude"))
print("Combined shape:", ocean["sst"].shape)

# %%
# ### 2.1 เตรียม SST บนกริด 0.5° (ลดความละเอียดจาก 0.25° เป็นทุก 2 จุด)
dates_all = pd.DatetimeIndex(pd.to_datetime(ocean[time_name].values).normalize())

sst_model = (
    ocean["sst"]
    .transpose(time_name, lat_name, lon_name)
    .to_numpy()
    .astype(np.float32)[:, ::2, ::2]
)
target_lat = ocean[lat_name].values[::2]
target_lon = ocean[lon_name].values[::2]
T, H, W = sst_model.shape

split_idx = int(dates_all.searchsorted(TEST_START))
if split_idx >= T or dates_all[split_idx] != TEST_START:
    raise ValueError(f"{TEST_START.date()} is missing from the SST time index.")

# ocean_mask: เซลล์ที่เป็นทะเล (มีข้อมูล ≥ 95% ในช่วง Train เท่านั้น)
ocean_mask = np.isfinite(sst_model[:split_idx]).mean(axis=0) >= 0.95

print("✅ sst_model :", sst_model.shape)
print("✅ ocean_mask:", ocean_mask.shape, f"({ocean_mask.sum()} ocean cells)")
print("✅ split_idx :", split_idx, f"-> test starts {dates_all[split_idx].date()}")



Using cache: 2017
Using cache: 2018
Using cache: 2019
Using cache: 2020
Using cache: 2021
Combined shape: (1826, 180, 240)
✅ sst_model : (1826, 90, 120)
✅ ocean_mask: (90, 120) (8515 ocean cells)
✅ split_idx : 1461 -> test starts 2021-01-01


In [3]:

# ## 3. โหลดตัวแปรบรรยากาศ (รายเดือน → ปรับให้เป็นรายวันบนกริด SST)
#
# ข้อควรรู้: ข้อมูลรายเดือนมี timestamp ที่วันที่ 1 ของเดือน
# ถ้า interpolate ไปวันหลังจาก timestamp สุดท้ายจะได้ NaN
# จึงอ่านเผื่อไว้ 1 เดือนหน้า-หลัง และตัดพื้นที่แบบเผื่อขอบ 5° (กัน NaN ที่ขอบกริด)

PAD_START = dates_all[0] - pd.offsets.MonthBegin(1)
PAD_END = dates_all[-1] + pd.offsets.MonthBegin(1)
AREA_MARGIN = 5.0


def load_monthly_as_daily(filename, var):
    """อ่านตัวแปรรายเดือน -> interpolate เข้ากริด SST (0.5°) -> รายวัน, คืน (T, H, W)"""
    with xr.open_dataset(RAW_DIR / filename) as ds:
        da = ds[var]
        da = da.sel(
            time=slice(PAD_START, PAD_END),
            lat=axis_slice(da["lat"].values, LAT_MIN - AREA_MARGIN, LAT_MAX + AREA_MARGIN),
            lon=axis_slice(da["lon"].values, LON_MIN - AREA_MARGIN, LON_MAX + AREA_MARGIN),
        ).load()
    da = da.interp(lat=target_lat, lon=target_lon, method="linear")
    da = da.interp(time=dates_all, method="linear")
    return da.transpose("time", "lat", "lon").to_numpy().astype(np.float32)


atm_raw = {
    "u": load_monthly_as_daily("uwnd.mon.mean.nc", "uwnd"),
    "v": load_monthly_as_daily("vwnd.mon.mean.nc", "vwnd"),
    "slp": load_monthly_as_daily("slp.mon.mean.nc", "slp"),
    "precip": load_monthly_as_daily("precip.mon.mean.nc", "precip"),
}
for name, arr in atm_raw.items():
    assert arr.shape == sst_model.shape, f"{name}: {arr.shape} != {sst_model.shape}"

# ### 3.1 ตรวจคุณภาพข้อมูลบรรยากาศ (รวมการตรวจทุกตัวแปรไว้ที่เดียว)
def contiguous_ranges(dates):
    """รวมวันที่ต่อเนื่องกันเป็นช่วง [(start, end), ...]"""
    ranges = []
    for d in dates:
        if ranges and (d - ranges[-1][1]).days == 1:
            ranges[-1][1] = d
        else:
            ranges.append([d, d])
    return ranges


# วันหนึ่งถือว่า "ใช้ได้" เมื่อทุกตัวแปรมีค่าครบทุกเซลล์ทะเล
valid_atm_day = np.ones(T, dtype=bool)
print("=== Atmospheric Data Check (เฉพาะเซลล์ทะเล) ===")
for name, arr in atm_raw.items():
    ocean_vals = arr[:, ocean_mask]
    finite = ocean_vals[np.isfinite(ocean_vals)]
    valid_atm_day &= np.isfinite(ocean_vals).all(axis=1)
    stats_txt = (
        f"min {finite.min():.2f} | max {finite.max():.2f} | mean {finite.mean():.2f}"
        if finite.size else "no valid data"
    )
    print(f"{name:7s} | NaN: {np.isnan(ocean_vals).mean():6.2%} | {stats_txt}")

bad_dates = dates_all[~valid_atm_day]
print(f"\nValid days: {int(valid_atm_day.sum())} / {T}  (missing: {len(bad_dates)})")
for start, end in contiguous_ranges(bad_dates):
    print(f"  missing {start.date()} to {end.date()} ({(end - start).days + 1} days)")



=== Atmospheric Data Check (เฉพาะเซลล์ทะเล) ===
u       | NaN:  0.00% | min -10.48 | max 10.51 | mean -2.05
v       | NaN:  0.00% | min -11.11 | max 8.35 | mean -0.18
slp     | NaN:  0.00% | min 1001.29 | max 1029.33 | mean 1011.19
precip  | NaN:  0.00% | min 0.00 | max 55.38 | mean 6.34

Valid days: 1826 / 1826  (missing: 0)


In [4]:

# ## 4. แบ่ง Train / Val / Test และ Normalize
#
# - **Test** = ปี 2021 ทั้งหมด (เฉพาะวันที่ข้อมูลบรรยากาศครบทั้ง window)
# - **Val** = 15% สุดท้ายของช่วงก่อน Test (เรียงตามเวลา)
# - ค่า mean/std คำนวณจากช่วง Train เท่านั้น เพื่อไม่ให้ข้อมูล Test รั่วเข้ามา

# %%
# window ใช้ได้เมื่อข้อมูลบรรยากาศครบทั้ง INPUT_DAYS วันก่อนวัน target
window_valid = np.zeros(T, dtype=bool)
for t in range(INPUT_DAYS, T):
    window_valid[t] = valid_atm_day[t - INPUT_DAYS:t].all()

usable = np.arange(INPUT_DAYS, T)
usable = usable[window_valid[usable]]
train_pool = usable[usable < split_idx]
test_targets = usable[usable >= split_idx]

if len(train_pool) < 20 or len(test_targets) == 0:
    raise ValueError("Too few valid windows.")

n_val = max(1, int(len(train_pool) * 0.15))
train_targets, val_targets = train_pool[:-n_val], train_pool[-n_val:]

# --- Normalize (fit เฉพาะ Train) ---
fit_end = int(train_targets[-1]) + 1


def fit_stats(arr):
    vals = arr[:fit_end][:, ocean_mask]
    vals = vals[np.isfinite(vals)]
    return float(vals.mean()), float(vals.std())


def normalize(arr, mean, std):
    return np.where(np.isfinite(arr), (arr - mean) / std, 0.0).astype(np.float32)


raw_variables = {"sst": sst_model, **atm_raw}
stats = {name: fit_stats(arr) for name, arr in raw_variables.items()}
norm = {name: normalize(arr, *stats[name]) for name, arr in raw_variables.items()}

sst_mean, sst_std = stats["sst"]
sst_norm = norm["sst"]

print("=== Normalization stats (Train only) ===")
for name, (m, s) in stats.items():
    print(f"{name:7s} mean={m:9.4f}  std={s:8.4f}")
print(f"\n✅ Train targets: {len(train_targets)}")
print(f"✅ Val targets  : {len(val_targets)}")
print(f"✅ Test targets : {len(test_targets)}")


# ### 4.1 คลังข้อมูลที่โมเดลเลือกใช้ได้ + ค่าอ้างอิงสำหรับประเมินผล
# แต่ละตัวแปรมี shape (T, H, W, C)
data_dict = {
    "sst": norm["sst"][..., None],
    "wind": np.stack([norm["u"], norm["v"]], axis=-1),
    "slp": norm["slp"][..., None],
    "precip": norm["precip"][..., None],
}
CHANNELS = {"sst": ["sst"], "wind": ["u", "v"], "slp": ["slp"], "precip": ["precip"]}

# ตัวแปรที่ใช้เป็น target ได้ (ทำนาย delta ของวันถัดไป)
target_series = {"sst": norm["sst"], "slp": norm["slp"]}

# ค่าจริง / Persistence / mask สำหรับประเมิน (ใช้ร่วมกันทุกโมเดล = เทียบกันแฟร์)
actual_test = sst_model[test_targets]
persistence_test = sst_model[test_targets - 1]
eval_mask = np.isfinite(actual_test) & np.isfinite(persistence_test) & ocean_mask[None]


def score(actual, predicted, mask):
    err = predicted[mask] - actual[mask]
    return float(np.mean(np.abs(err))), float(np.sqrt(np.mean(err ** 2)))


=== Normalization stats (Train only) ===
sst     mean=  27.6614  std=  3.1940
u       mean=  -1.9165  std=  3.2440
v       mean=  -0.2127  std=  2.4640
slp     mean=1011.3451  std=  3.1872
precip  mean=   6.1289  std=  3.8561

✅ Train targets: 1236
✅ Val targets  : 218
✅ Test targets : 365


In [7]:

# ## 5. Dataset, Model, Training utilities

# %%
def build_features(inputs):
    """รวมตัวแปรที่เลือกเป็น array เดียว (T, H, W, C)"""
    return np.concatenate([data_dict[k] for k in inputs], axis=-1)


class ForecastDataset(Dataset):
    """
    x : (INPUT_DAYS*C, H, W)  ข้อมูลย้อนหลัง (channel index = day*C + c)
    y : (n_targets, H, W)     delta ของวัน t เทียบกับ t-1 (normalized)
    w : (n_targets, H, W)     1 = จุดทะเลที่ใช้คำนวณ loss
    """

    def __init__(self, indices, features, target_arrays):
        self.indices = np.asarray(indices, dtype=int)
        self.features = features
        self.target_arrays = target_arrays

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        t = self.indices[i]
        window = self.features[t - INPUT_DAYS:t]                  # (D, H, W, C)
        x = window.transpose(0, 3, 1, 2).reshape(-1, H, W)       # (D*C, H, W)

        valid = np.isfinite(sst_model[t]) & np.isfinite(sst_model[t - 1]) & ocean_mask
        y = np.stack([a[t] - a[t - 1] for a in self.target_arrays])
        y = np.where(valid, y, 0.0).astype(np.float32)
        w = np.broadcast_to(valid, y.shape).astype(np.float32)

        return (
            torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)),
            torch.from_numpy(y),
            torch.from_numpy(w),
        )


class ResidualCNN(nn.Module):
    """CNN ทำนาย delta; layer สุดท้ายเริ่มที่ 0 จึงเริ่มต้นเท่ากับ Persistence"""

    def __init__(self, in_channels, out_channels=1, hidden=(16, 16, 8)):
        super().__init__()
        layers, prev = [], in_channels
        for h in hidden:
            layers += [nn.Conv2d(prev, h, 3, padding=1), nn.ReLU()]
            prev = h
        head = nn.Conv2d(prev, out_channels, 1)
        nn.init.zeros_(head.weight)
        nn.init.zeros_(head.bias)
        layers.append(head)
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ### 5.1 Physics-informed term
# หมายเหตุ: นี่เป็น regularizer เชิงฮิวริสติก ไม่ใช่สมการ advection เต็มรูปแบบ
# (ลงโทษไม่ให้ delta ที่ทำนายเปลี่ยนแปลงไปตามทิศลม u·∇(ΔT) → 0)
def physics_config(inputs, weight):
    """หา index ของช่องลม u, v ของ 'วันสุดท้าย' ใน tensor input"""
    names = [c for k in inputs for c in CHANNELS[k]]
    last_day_offset = (INPUT_DAYS - 1) * len(names)
    return {
        "u": last_day_offset + names.index("u"),
        "v": last_day_offset + names.index("v"),
        "weight": weight,
    }


def physics_loss(preds, bx, cfg):
    u, v = bx[:, cfg["u"]], bx[:, cfg["v"]]
    d_dy, d_dx = torch.gradient(preds[:, 0], dim=(1, 2))
    return ((u * d_dx + v * d_dy) ** 2).mean()


# ### 5.2 Training / Prediction
def run_epoch(model, loader, optimizer=None, physics=None):
    """รัน 1 epoch; คืนค่า data loss (MSE ถ่วงด้วย mask) เพื่อให้ train/val เทียบกันได้"""
    training = optimizer is not None
    model.train(training)
    total, count = 0.0, 0
    with torch.set_grad_enabled(training):
        for bx, by, bw in loader:
            bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
            preds = model(bx)
            data_loss = (((preds - by) ** 2) * bw).mean()

            if training:
                loss = data_loss
                if physics is not None:
                    loss = loss + physics["weight"] * physics_loss(preds, bx, physics)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total += data_loss.item() * bx.size(0)
            count += bx.size(0)
    return total / count


def train_model(model, train_loader, val_loader, name, physics=None,
                epochs=EPOCHS, patience=PATIENCE):
    """เทรนพร้อม early stopping แล้วโหลด weight ที่ val ดีที่สุดกลับเข้าโมเดล"""
    optimizer = optim.Adam(model.parameters(), lr=LR)
    best_val, best_state, wait = float("inf"), copy.deepcopy(model.state_dict()), 0

    for epoch in range(1, epochs + 1):
        train_loss = run_epoch(model, train_loader, optimizer, physics)
        val_loss = run_epoch(model, val_loader)
        print(f"Epoch {epoch:02d}/{epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

        if val_loss < best_val:
            best_val, wait = val_loss, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= patience:
                print(f"⏹️ Early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    safe_name = "".join(c if c.isalnum() else "_" for c in name)
    torch.save(best_state, MODEL_DIR / f"{safe_name}.pth")
    return best_val


@torch.no_grad()
def predict_sst_delta(model, dataset):
    """คืน delta ของ SST (channel แรก) แบบ normalized, shape (N, H, W)"""
    model.eval()
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    return np.concatenate([model(bx.to(DEVICE))[:, 0].cpu().numpy() for bx, _, _ in loader])


def run_experiment(name, inputs, targets=("sst",), physics_weight=0.0,
                   hidden=(16, 16, 8), epochs=EPOCHS):
    """เทรน 1 การทดลอง แล้วประเมินบน Test set; คืน (model, metrics)"""
    print(f"\n🚀 {name} | inputs={inputs} | targets={targets}")
    torch.manual_seed(SEED)

    features = build_features(inputs)
    target_arrays = [target_series[k] for k in targets]
    train_ds = ForecastDataset(train_targets, features, target_arrays)
    val_ds = ForecastDataset(val_targets, features, target_arrays)
    test_ds = ForecastDataset(test_targets, features, target_arrays)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = ResidualCNN(features.shape[-1] * INPUT_DAYS, len(targets), hidden).to(DEVICE)
    physics = physics_config(inputs, physics_weight) if physics_weight > 0 else None
    train_model(model, train_loader, val_loader, name, physics, epochs)

    delta = predict_sst_delta(model, test_ds)
    pred_sst = (sst_norm[test_targets - 1] + delta) * sst_std + sst_mean
    mae, rmse = score(actual_test, pred_sst, eval_mask)
    print(f"✅ {name} | MAE: {mae:.4f} | RMSE: {rmse:.4f}")
    return model, {"Experiment": name, "MAE": mae, "RMSE": rmse}



In [ ]:

# ## 6. Ablation Study (E1–E6)
#
# ทุกการทดลองใช้ train/val/test ชุดเดียวกัน, seed เดียวกัน, epoch และ early stopping เท่ากัน
#
# | รหัส | Input | หมายเหตุ |
# |------|-------|----------|
# | E1 | SST | baseline ของ CNN |
# | E2 | SST + Wind | |
# | E3 | SST + SLP | |
# | E4 | SST + Wind + SLP | |
# | E5 | SST + Wind + SLP | Multi-task (ทำนาย SST + SLP) + Physics loss |
# | E6 | SST + Precipitation | |

# %%
import copy
EXPERIMENTS = [
    dict(name="E1: SST Baseline", inputs=("sst",)),
    dict(name="E2: SST + Wind", inputs=("sst", "wind")),
    dict(name="E3: SST + SLP", inputs=("sst", "slp")),
    dict(name="E4: SST + Wind + SLP", inputs=("sst", "wind", "slp")),
    dict(name="E5: Physics Multi-Task (SST + Wind + SLP)",
         inputs=("sst", "wind", "slp"), targets=("sst", "slp"),
         physics_weight=0.1, hidden=(32, 32, 16)),
    dict(name="E6: SST + Precipitation", inputs=("sst", "precip")),
]

trained_models = {}
rows = []
base_mae, base_rmse = score(actual_test, persistence_test, eval_mask)
rows.append({"Experiment": "Persistence", "MAE": base_mae, "RMSE": base_rmse})

for cfg in EXPERIMENTS:
    model, metrics = run_experiment(**cfg)
    trained_models[cfg["name"]] = model
    rows.append(metrics)


# ### 6.1 สรุปผล
results_df = pd.DataFrame(rows)
e1_mae = results_df.loc[results_df["Experiment"] == "E1: SST Baseline", "MAE"].item()
e1_rmse = results_df.loc[results_df["Experiment"] == "E1: SST Baseline", "RMSE"].item()
results_df["MAE gain vs E1 (%)"] = (e1_mae - results_df["MAE"]) / e1_mae * 100
results_df["RMSE gain vs E1 (%)"] = (e1_rmse - results_df["RMSE"]) / e1_rmse * 100

print(f"=== Ablation Study Results (Test: 2021, {len(test_targets)} days) ===")
display(results_df.round(4))
results_df.to_csv(REPORT_DIR / "ablation_results.csv", index=False)
print("✅ Saved: reports/ablation_results.csv")

# ### 6.2 กราฟเปรียบเทียบ
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(results_df))
width = 0.38.venv

mae_bars = ax.bar(x - width / 2, results_df["MAE"], width, label="MAE (°C)", color="skyblue")
rmse_bars = ax.bar(x + width / 2, results_df["RMSE"], width, label="RMSE (°C)", color="salmon")
for bars in (mae_bars, rmse_bars):
    ax.bar_label(bars, fmt="%.4f", fontsize=8, padding=2)

ax.set_ylabel("Error (°C)")
ax.set_title("Ablation Study: Impact of Atmospheric Variables on SST Forecasting")
ax.set_xticks(x)
ax.set_xticklabels(results_df["Experiment"], rotation=20, ha="right")
ax.legend()
plt.tight_layout()
plt.savefig(REPORT_DIR / "ablation_study_results.png", dpi=150)
plt.show()
print("✅ Saved: reports/ablation_study_results.png")



🚀 E1: SST Baseline | inputs=('sst',) | targets=('sst',)
Epoch 01/15 | Train Loss: 0.004526 | Val Loss: 0.003009
Epoch 02/15 | Train Loss: 0.004396 | Val Loss: 0.002958
Epoch 03/15 | Train Loss: 0.004342 | Val Loss: 0.002906
Epoch 04/15 | Train Loss: 0.004291 | Val Loss: 0.002977
Epoch 05/15 | Train Loss: 0.004279 | Val Loss: 0.002919
Epoch 06/15 | Train Loss: 0.004255 | Val Loss: 0.002831
Epoch 07/15 | Train Loss: 0.004232 | Val Loss: 0.002827
Epoch 08/15 | Train Loss: 0.004229 | Val Loss: 0.002913
Epoch 09/15 | Train Loss: 0.004216 | Val Loss: 0.002813
Epoch 10/15 | Train Loss: 0.004202 | Val Loss: 0.002763
Epoch 11/15 | Train Loss: 0.004197 | Val Loss: 0.002770
Epoch 12/15 | Train Loss: 0.004188 | Val Loss: 0.002781
Epoch 13/15 | Train Loss: 0.004181 | Val Loss: 0.002833
⏹️ Early stopping at epoch 13
✅ E1: SST Baseline | MAE: 0.1184 | RMSE: 0.1812

🚀 E2: SST + Wind | inputs=('sst', 'wind') | targets=('sst',)
Epoch 01/15 | Train Loss: 0.004568 | Val Loss: 0.003032
Epoch 02/15 | Train

In [ ]:

# ## 7. Multi-step Forecast (Recursive)
#
# ใช้โมเดล SST-only (E1) ทำนายวันถัดไป แล้วป้อนผลกลับเข้า input เพื่อทำนายต่อ
# จุดเริ่มต้นพยากรณ์ใช้ทุกวันของปี 2021 ที่ยังเหลือ SST จริงให้เทียบครบ `max(EVAL_HORIZONS)` วัน

# %%
@torch.no_grad()
def recursive_forecast(model, origins, max_horizon, batch_size=16):
    """คืน {horizon: array (N, H, W) หน่วย °C}; horizon h = ทำนายวัน origin + h - 1"""
    model.eval()
    outputs = {h: [] for h in range(1, max_horizon + 1)}

    for start in range(0, len(origins), batch_size):
        batch = origins[start:start + batch_size]
        history = np.stack([sst_norm[t - INPUT_DAYS:t] for t in batch]).astype(np.float32)  # (N, D, H, W)

        for h in range(1, max_horizon + 1):
            delta = model(torch.from_numpy(history).to(DEVICE))[:, 0].cpu().numpy()
            next_norm = history[:, -1] + delta
            next_norm[:, ~ocean_mask] = 0.0                       # ปิดพื้นที่ที่ไม่ใช่ทะเล
            outputs[h].append(next_norm * sst_std + sst_mean)     # แปลงกลับเป็น °C
            history = np.concatenate([history[:, 1:], next_norm[:, None]], axis=1)

    return {h: np.concatenate(v, axis=0).astype(np.float32) for h, v in outputs.items()}


max_horizon = max(EVAL_HORIZONS)
origins = np.arange(split_idx, T - max_horizon + 1)
print("Forecast origins:", len(origins), "|", dates_all[origins[0]].date(), "to", dates_all[origins[-1]].date())

forecasts = recursive_forecast(trained_models["E1: SST Baseline"], origins, max_horizon)
print("✅ Recursive forecasting completed.")


horizon_rows = []
for h in EVAL_HORIZONS:
    actual = sst_model[origins + h - 1]
    persistence = sst_model[origins - 1]
    mask = np.isfinite(actual) & np.isfinite(persistence) & ocean_mask[None]

    p_mae, p_rmse = score(actual, persistence, mask)
    m_mae, m_rmse = score(actual, forecasts[h], mask)
    horizon_rows.append({
        "Forecast Horizon (days)": h,
        "Persistence MAE": p_mae,
        "CNN MAE": m_mae,
        "MAE Improvement (%)": (p_mae - m_mae) / p_mae * 100,
        "Persistence RMSE": p_rmse,
        "CNN RMSE": m_rmse,
        "RMSE Improvement (%)": (p_rmse - m_rmse) / p_rmse * 100,
    })

horizon_df = pd.DataFrame(horizon_rows)
display(horizon_df.round(4))
horizon_df.to_csv(REPORT_DIR / "forecast_horizon_results.csv", index=False)
print("✅ Saved: reports/forecast_horizon_results.csv")


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, ("MAE", "RMSE")):
    ax.plot(horizon_df["Forecast Horizon (days)"], horizon_df[f"Persistence {metric}"],
            marker="o", label="Persistence", color="gray", linewidth=2)
    ax.plot(horizon_df["Forecast Horizon (days)"], horizon_df[f"CNN {metric}"],
            marker="s", label="Residual CNN (SST-only)", color="blue", linewidth=2)
    ax.set_xlabel("Forecast Horizon (days)")
    ax.set_ylabel(f"{metric} (°C)")
    ax.set_title(f"{metric} Comparison ({', '.join(map(str, EVAL_HORIZONS))} days)")
    ax.set_xticks(list(EVAL_HORIZONS))
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(REPORT_DIR / "forecast_horizon_comparison.png", dpi=150)
plt.show()
print("✅ Saved: reports/forecast_horizon_comparison.png")

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("/home/kankrit/Projects/ocean-forecasting/data/raw")

def load_new_atmospheric_data(filename, var_name):
    with xr.open_dataset(RAW_DIR / filename) as ds:
        # 🔧 ใช้ axis_slice เพื่อจัดการทิศทางของ lat/lon ให้ถูกต้อง
        da = ds[var_name].sel(
            time=slice("2017-01-01", "2021-12-31"),
            lat=axis_slice(ds["lat"].values, LAT_MIN, LAT_MAX),
            lon=axis_slice(ds["lon"].values, LON_MIN, LON_MAX)
        ).load()
    
    # Interpolate พื้นที่
    da = da.interp(lat=target_lat, lon=target_lon, method="linear")
    
    # Interpolate เวลา
    da = da.interp(time=target_dates, method="linear").load()
    
    return da.transpose("time", "lat", "lon").to_numpy().astype(np.float32)

# ลองโหลด Air Temp ใหม่
air_array = load_new_atmospheric_data("air.2m.mon.mean.nc", "air")

# ถ้าสำเร็จ จะเห็น Shape
print("✅ Air array shape:", air_array.shape)

In [ ]:
# ใช้ precip_norm ที่มีอยู่แล้ว (จาก CMAP)
data_dict.update({
    "precip": precip_norm[..., np.newaxis]  # มีอยู่แล้ว
})

# รันการทดลอง E1, E7 (ฝน)
results_v2 = []

# E1: Baseline
mae, rmse = train_ablation_model("E1: SST Baseline", ["sst"])
results_v2.append({"Experiment": "E1: SST Baseline", "MAE": mae, "RMSE": rmse})

# E7: SST + Precipitation (ฝน)
mae, rmse = train_ablation_model("E7: SST + Precipitation", ["sst", "precip"])
results_v2.append({"Experiment": "E7: SST + Precip", "MAE": mae, "RMSE": rmse})

results_v2_df = pd.DataFrame(results_v2)
display(results_v2_df.round(4))

In [ ]:
import torch.nn as nn
import torch.optim as optim
import copy

# ---------------------------------------------------------
# 1. สร้างโมเดลแบบ Shared Encoder + Multiple Heads
# ---------------------------------------------------------
class MultiTaskCNN(nn.Module):
    def __init__(self, in_channels, hidden=32):
        super().__init__()
        # Shared Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 3, padding="same"), nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding="same"), nn.ReLU(),
        )
        # Head 1: SST
        self.sst_head = nn.Sequential(
            nn.Conv2d(hidden, 8, 3, padding="same"), nn.ReLU(),
            nn.Conv2d(8, 1, 1, padding="same")
        )
        # Head 2: SLP
        self.slp_head = nn.Sequential(
            nn.Conv2d(hidden, 8, 3, padding="same"), nn.ReLU(),
            nn.Conv2d(8, 1, 1, padding="same")
        )
        nn.init.zeros_(self.sst_head[-1].weight); nn.init.zeros_(self.sst_head[-1].bias)
        nn.init.zeros_(self.slp_head[-1].weight); nn.init.zeros_(self.slp_head[-1].bias)

    def forward(self, x):
        features = self.encoder(x)
        return self.sst_head(features), self.slp_head(features)

# ---------------------------------------------------------
# 2. เตรียม Input (SST + Wind + SLP)
# ---------------------------------------------------------
inputs = ["sst", "wind", "slp"]
features = np.concatenate([data_dict[k] for k in inputs], axis=-1)
in_channels = features.shape[-1] * input_days

# สร้าง Dataset
class MultiTargetDataset(Dataset):
    def __init__(self, indices, features):
        self.indices = np.asarray(indices, dtype=int)
        self.features = features

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        t = self.indices[i]
        window = self.features[t - input_days:t]
        x = window.transpose(0, 3, 1, 2).reshape(-1, H, W)

        valid = np.isfinite(sst_model[t]) & np.isfinite(sst_model[t - 1]) & ocean_mask
        # Target: SST และ SLP Delta
        y = np.stack([
            sst_norm[t] - sst_norm[t - 1],
            slp_norm[t] - slp_norm[t - 1]
        ])
        y = np.where(valid, y, 0.0).astype(np.float32)
        w = np.broadcast_to(valid, y.shape).astype(np.float32)

        return (
            torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)),
            torch.from_numpy(y),
            torch.from_numpy(w),
        )

train_ds = MultiTargetDataset(train_targets, features)
val_ds = MultiTargetDataset(val_targets, features)
test_ds = MultiTargetDataset(test_targets, features)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

# ---------------------------------------------------------
# 3. เทรนแบบ Multi-Task (ปรับ Loss Weight)
# ---------------------------------------------------------
torch.manual_seed(SEED)
model = MultiTaskCNN(in_channels).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss(reduction='none')

LAMBDA_SLP = 0.1  # 🔧 ลดความสำคัญของ SLP ลง
EPOCHS_MT = 15
PATIENCE_MT = 3

best_val_loss = float('inf')
best_state = copy.deepcopy(model.state_dict())
wait = 0

print("\n🚀 เริ่มเทรน Multi-Task V2 (SST + SLP)")
for epoch in range(EPOCHS_MT):
    # --- Train ---
    model.train()
    for bx, by, bw in train_loader:
        bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
        sst_pred, slp_pred = model(bx)
        
        sst_target = by[:, 0:1] 
        slp_target = by[:, 1:2] 
        sst_w = bw[:, 0:1]
        slp_w = bw[:, 1:2]
        
        loss_sst = (((sst_pred - sst_target) ** 2) * sst_w).mean()
        loss_slp = (((slp_pred - slp_target) ** 2) * slp_w).mean()
        
        total_loss = loss_sst + LAMBDA_SLP * loss_slp
        
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

    # --- Validation (คำนวณเฉพาะ SST) ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for bx, by, bw in val_loader:
            bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
            sst_pred, _ = model(bx)
            sst_w = bw[:, 0:1]
            val_loss += (((sst_pred - by[:, 0:1]) ** 2) * sst_w).mean().item() * bx.size(0)
    val_loss /= len(val_loader.dataset)
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS_MT} | Val SST Loss: {val_loss:.6f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE_MT:
            print(f"⏹️ Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(best_state)
print("✅ เทรน Multi-Task V2 เสร็จสิ้น!")

# ---------------------------------------------------------
# 4. ประเมินผล
# ---------------------------------------------------------
model.eval()
predictions = []
with torch.no_grad():
    for bx, _, _ in DataLoader(test_ds, batch_size=4, shuffle=False):
        sst_pred, _ = model(bx.to(DEVICE))
        predictions.append(sst_pred.cpu().numpy()[:, 0])

delta_pred = np.concatenate(predictions, axis=0)
pred_sst = (sst_norm[test_targets - 1] + delta_pred) * sst_std + sst_mean

mae_mt, rmse_mt = score(actual_test, pred_sst, eval_mask)
print(f"\n✅ Multi-Task V2 (Lambda={LAMBDA_SLP}) | MAE: {mae_mt:.4f} | RMSE: {rmse_mt:.4f}")

# เทียบกับ E1
mae_e1, rmse_e1 = score(actual_test, sst_model[test_targets - 1] + (0 * delta_pred), eval_mask) # Persistence
# ควรเอา MAE ของ E1 จากรอบที่แล้วมาเทียบ
print(f"📊 เทียบกับ E1 (SST only) จากรอบก่อน: MAE {0.1205:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy

# ---------------------------------------------------------
# 1. ฟังก์ชันคำนวณ Physics Loss (Advection)
# ---------------------------------------------------------
def physics_loss(pred_delta_sst, u_wind, v_wind):
    # หา Gradient ของ SST Delta ในแนวแกน x (dx) และ y (dy)
    dT_dy, dT_dx = torch.gradient(pred_delta_sst[:, 0], dim=(1, 2))
    
    # คำนวณ Advection
    advection = u_wind[:, 0] * dT_dx + v_wind[:, 0] * dT_dy
    
    # บังคับให้ advection ใกล้ 0 (สมดุล)
    return torch.mean(advection ** 2)

# ---------------------------------------------------------
# 2. เตรียม Dataset (SST + Wind)
# ---------------------------------------------------------
class PhysicsDataset(Dataset):
    def __init__(self, indices, features):
        self.indices = np.asarray(indices, dtype=int)
        self.features = features

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        t = self.indices[i]
        window = self.features[t - input_days:t]
        x = window.transpose(0, 3, 1, 2).reshape(-1, H, W)

        valid = np.isfinite(sst_model[t]) & np.isfinite(sst_model[t - 1]) & ocean_mask
        y = sst_norm[t] - sst_norm[t - 1]
        y = np.where(valid, y, 0.0).astype(np.float32)
        w = valid.astype(np.float32)

        return (
            torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)),
            torch.from_numpy(np.expand_dims(y, 0)),
            torch.from_numpy(np.expand_dims(w, 0)),
        )

# สร้าง Input (SST + Wind)
inputs_phys = ["sst", "wind"]
features_phys = np.concatenate([data_dict[k] for k in inputs_phys], axis=-1)
in_channels_phys = features_phys.shape[-1] * input_days

train_ds_phys = PhysicsDataset(train_targets, features_phys)
val_ds_phys = PhysicsDataset(val_targets, features_phys)
test_ds_phys = PhysicsDataset(test_targets, features_phys)

train_loader_phys = DataLoader(train_ds_phys, batch_size=4, shuffle=True)
val_loader_phys = DataLoader(val_ds_phys, batch_size=4, shuffle=False)

# ---------------------------------------------------------
# 3. สร้างโมเดล (ใช้ ResidualCNN ตัวเดิม)
# ---------------------------------------------------------
class PhysicsCNN(nn.Module):
    def __init__(self, in_channels, hidden=(32, 32, 16)):
        super().__init__()
        layers, prev = [], in_channels
        for h in hidden:
            layers += [nn.Conv2d(prev, h, 3, padding=1), nn.ReLU()]
            prev = h
        head = nn.Conv2d(prev, 1, 1)
        nn.init.zeros_(head.weight); nn.init.zeros_(head.bias)
        layers.append(head)
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

torch.manual_seed(SEED)
phys_model = PhysicsCNN(in_channels_phys).to(DEVICE)
optimizer = optim.Adam(phys_model.parameters(), lr=1e-3)
criterion = nn.MSELoss(reduction='none')

# 🔧 น้ำหนักของ Physics Loss (สำคัญมาก! ถ้ามากเกินโมเดลจะทำนายแย่)
LAMBDA_PHYSICS = 0.01 

EPOCHS_PHYS = 15
PATIENCE_PHYS = 3
best_val_loss = float('inf')
best_state_phys = copy.deepcopy(phys_model.state_dict())
wait = 0

print("\n🚀 เริ่มเทรน Physics-Informed Model (SST + Wind)")
for epoch in range(EPOCHS_PHYS):
    # --- Train ---
    phys_model.train()
    train_loss_total = 0.0
    for bx, by, bw in train_loader_phys:
        bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
        
        # ดึง Wind U และ V ของวันสุดท้ายออกมา (สำหรับคำนวณ Physics)
        # ลำดับ Channels: [SST(7), U(7), V(7)]
        # วันสุดท้ายคือ index: -1 (V), -2 (U)
        u_last = bx[:, -2, :, :].unsqueeze(1)
        v_last = bx[:, -1, :, :].unsqueeze(1)
        
        preds = phys_model(bx)
        
        # 1. Data Loss
        data_loss = (((preds - by) ** 2) * bw).mean()
        
        # 2. Physics Loss
        p_loss = physics_loss(preds, u_last, v_last)
        
        # 3. Total Loss
        total_loss = data_loss + LAMBDA_PHYSICS * p_loss
        
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        train_loss_total += total_loss.item() * bx.size(0)
        
    train_loss_total /= len(train_loader_phys.dataset)

    # --- Validation ---
    phys_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for bx, by, bw in val_loader_phys:
            bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
            preds = phys_model(bx)
            val_loss += (((preds - by) ** 2) * bw).mean().item() * bx.size(0)
    val_loss /= len(val_loader_phys.dataset)
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS_PHYS} | Train Loss: {train_loss_total:.6f} | Val SST Loss: {val_loss:.6f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_phys = copy.deepcopy(phys_model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE_PHYS:
            print(f"⏹️ Early stopping at epoch {epoch+1}")
            break

phys_model.load_state_dict(best_state_phys)
print("✅ เทรน Physics-Informed Model เสร็จสิ้น!")

# ---------------------------------------------------------
# 4. ประเมินผล
# ---------------------------------------------------------
phys_model.eval()
predictions = []
with torch.no_grad():
    for bx, _, _ in DataLoader(test_ds_phys, batch_size=4, shuffle=False):
        preds = phys_model(bx.to(DEVICE))
        predictions.append(preds.cpu().numpy()[:, 0])

delta_pred = np.concatenate(predictions, axis=0)
pred_sst = (sst_norm[test_targets - 1] + delta_pred) * sst_std + sst_mean

mae_phys, rmse_phys = score(actual_test, pred_sst, eval_mask)
print(f"\n✅ Physics-Informed Model | MAE: {mae_phys:.4f} | RMSE: {rmse_phys:.4f}")

print("\n=== สรุปผลลัพธ์ ===")
print(f"1. E1 Baseline (SST)      | MAE: 0.1205 | RMSE: 0.1836")
print(f"2. Multi-Task V2 (SST+SLP)| MAE: 0.1203 | RMSE: 0.1828")
print(f"3. Physics (SST+Wind)     | MAE: {mae_phys:.4f} | RMSE: {rmse_phys:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# รวมผลลัพธ์ V2
results_v2 = pd.DataFrame([
    {"Model": "Persistence", "MAE": 0.1243, "RMSE": 0.1919},
    {"Model": "E1: SST Baseline (V1)", "MAE": 0.1205, "RMSE": 0.1836},
    {"Model": "E2: SST + Wind (V1)", "MAE": 0.1238, "RMSE": 0.1871},
    {"Model": "Multi-Task V2 (SST+SLP)", "MAE": 0.1203, "RMSE": 0.1828},
    {"Model": "Physics-Informed V2", "MAE": 0.1195, "RMSE": 0.1811},
])

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_v2))
width = 0.35

bars1 = ax.bar(x - width/2, results_v2["MAE"], width, label='MAE (°C)', color='skyblue')
bars2 = ax.bar(x + width/2, results_v2["RMSE"], width, label='RMSE (°C)', color='salmon')

ax.set_ylabel('Error (°C)')
ax.set_title('Model Evolution: V1 Baseline vs V2 Advanced Models')
ax.set_xticks(x)
ax.set_xticklabels(results_v2["Model"], rotation=20, ha='right')
ax.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.4f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig("reports/v2_model_comparison.png", dpi=150)
plt.show()

In [ ]:
import xarray as xr
import glob
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("/home/kankrit/Projects/ocean-forecasting/data/raw")

print("⏳ กำลังรวมไฟล์ ERA5 ทั้ง 60 ไฟล์...")
era5_files = sorted(glob.glob(str(RAW_DIR / "era5_wind_*.nc")))
datasets = [xr.open_dataset(f) for f in era5_files]
wind_era5 = xr.concat(datasets, dim="valid_time").sortby("valid_time")
for ds in datasets: ds.close()
print("✅ รวมไฟล์สำเร็จ! Shape:", dict(wind_era5.sizes))

# 🔧 1. เปลี่ยนชื่อมิติจาก latitude/longitude เป็น lat/lon และ valid_time เป็น time
wind_era5 = wind_era5.rename({
    "latitude": "lat",
    "longitude": "lon",
    "valid_time": "time"
})

# 🔧 2. Interpolate พื้นที่ให้ตรงกับ SST
wind_era5_grid = wind_era5.interp(lat=target_lat, lon=target_lon, method="linear")

# 🔧 3. Interpolate เวลาให้ตรงกับ SST
target_dates = pd.DatetimeIndex(pd.to_datetime(ocean[time_name].values).normalize())
wind_era5_daily = wind_era5_grid.interp(time=target_dates, method="linear").load()

# 🔧 4. แปลงเป็น numpy array (ใช้ชื่อตัวแปร u10/v10 ที่ ERA5 ใช้)
u_era5 = wind_era5_daily["u10"].to_numpy().astype(np.float32)
v_era5 = wind_era5_daily["v10"].to_numpy().astype(np.float32)

print("✅ u_era5 shape:", u_era5.shape)
print("✅ v_era5 shape:", v_era5.shape)

In [ ]:
# Normalize ด้วย Mean/Std จาก Train set
u_era5_mean = float(np.nanmean(u_era5[:split_idx][:, ocean_mask]))
u_era5_std = float(np.nanstd(u_era5[:split_idx][:, ocean_mask]))
u_era5_norm = np.where(np.isfinite(u_era5), (u_era5 - u_era5_mean) / u_era5_std, 0.0).astype(np.float32)

v_era5_mean = float(np.nanmean(v_era5[:split_idx][:, ocean_mask]))
v_era5_std = float(np.nanstd(v_era5[:split_idx][:, ocean_mask]))
v_era5_norm = np.where(np.isfinite(v_era5), (v_era5 - v_era5_mean) / v_era5_std, 0.0).astype(np.float32)

# อัปเดต data_dict (เพิ่ม wind_era5)
data_dict.update({
    "wind_era5": np.stack([u_era5_norm, v_era5_norm], axis=-1)
})

print("✅ อัปเดต data_dict สำเร็จ! (มี wind_era5 เพิ่มมา)")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy

# --- ฟังก์ชัน Physics Loss ---
def physics_loss(pred_delta_sst, u_wind, v_wind):
    dT_dy, dT_dx = torch.gradient(pred_delta_sst[:, 0], dim=(1, 2))
    advection = u_wind[:, 0] * dT_dx + v_wind[:, 0] * dT_dy
    return torch.mean(advection ** 2)

# --- Dataset ---
class PhysicsDataset(Dataset):
    def __init__(self, indices, features):
        self.indices = np.asarray(indices, dtype=int)
        self.features = features
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        t = self.indices[i]
        window = self.features[t - input_days:t]
        x = window.transpose(0, 3, 1, 2).reshape(-1, H, W)
        valid = np.isfinite(sst_model[t]) & np.isfinite(sst_model[t - 1]) & ocean_mask
        y = np.where(valid, sst_norm[t] - sst_norm[t - 1], 0.0).astype(np.float32)
        w = valid.astype(np.float32)
        return (torch.from_numpy(np.ascontiguousarray(x)), 
                torch.from_numpy(np.expand_dims(y, 0)), 
                torch.from_numpy(np.expand_dims(w, 0)))

# --- สร้าง Input (SST + Wind ERA5) ---
inputs_era5 = ["sst", "wind_era5"]
features_era5 = np.concatenate([data_dict[k] for k in inputs_era5], axis=-1)
in_channels_era5 = features_era5.shape[-1] * input_days

train_ds = PhysicsDataset(train_targets, features_era5)
val_ds = PhysicsDataset(val_targets, features_era5)
test_ds = PhysicsDataset(test_targets, features_era5)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

# --- โมเดล ---
class PhysicsCNN(nn.Module):
    def __init__(self, in_channels, hidden=(32, 32, 16)):
        super().__init__()
        layers, prev = [], in_channels
        for h in hidden:
            layers += [nn.Conv2d(prev, h, 3, padding=1), nn.ReLU()]
            prev = h
        head = nn.Conv2d(prev, 1, 1)
        nn.init.zeros_(head.weight); nn.init.zeros_(head.bias)
        layers.append(head)
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

torch.manual_seed(SEED)
phys_model_era5 = PhysicsCNN(in_channels_era5).to(DEVICE)
optimizer = optim.Adam(phys_model_era5.parameters(), lr=1e-3)
criterion = nn.MSELoss(reduction='none')

LAMBDA_PHYSICS = 0.01
EPOCHS_PHYS = 15
PATIENCE_PHYS = 3
best_val_loss = float('inf')
best_state_phys = copy.deepcopy(phys_model_era5.state_dict())
wait = 0

print("\n🚀 เริ่มเทรน Physics-Informed Model (SST + ERA5 Wind)")
for epoch in range(EPOCHS_PHYS):
    phys_model_era5.train()
    for bx, by, bw in train_loader:
        bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
        u_last = bx[:, -2, :, :].unsqueeze(1)
        v_last = bx[:, -1, :, :].unsqueeze(1)
        preds = phys_model_era5(bx)
        data_loss = (((preds - by) ** 2) * bw).mean()
        p_loss = physics_loss(preds, u_last, v_last)
        total_loss = data_loss + LAMBDA_PHYSICS * p_loss
        optimizer.zero_grad(); total_loss.backward(); optimizer.step()

    phys_model_era5.eval()
    val_loss = 0.0
    with torch.no_grad():
        for bx, by, bw in val_loader:
            bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
            preds = phys_model_era5(bx)
            val_loss += (((preds - by) ** 2) * bw).mean().item() * bx.size(0)
    val_loss /= len(val_loader.dataset)
    print(f"Epoch {epoch+1:02d}/{EPOCHS_PHYS} | Val SST Loss: {val_loss:.6f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_phys = copy.deepcopy(phys_model_era5.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE_PHYS:
            print(f"⏹️ Early stopping at epoch {epoch+1}")
            break

phys_model_era5.load_state_dict(best_state_phys)
print("✅ เทรน Physics-Informed (ERA5) เสร็จสิ้น!")

# --- ประเมินผล ---
phys_model_era5.eval()
predictions = []
with torch.no_grad():
    for bx, _, _ in DataLoader(test_ds, batch_size=4, shuffle=False):
        preds = phys_model_era5(bx.to(DEVICE))
        predictions.append(preds.cpu().numpy()[:, 0])

delta_pred = np.concatenate(predictions, axis=0)
pred_sst = (sst_norm[test_targets - 1] + delta_pred) * sst_std + sst_mean
mae_era5, rmse_era5 = score(actual_test, pred_sst, eval_mask)

print(f"\n✅ Physics (ERA5) | MAE: {mae_era5:.4f} | RMSE: {rmse_era5:.4f}")

print("\n=== สรุปผลลัพธ์ ===")
print(f"1. SST Baseline          | MAE: 0.1205 | RMSE: 0.1836")
print(f"2. Multi-Task V2         | MAE: 0.1203 | RMSE: 0.1828")
print(f"3. Physics (NCEP Wind)   | MAE: 0.1195 | RMSE: 0.1811")
print(f"4. Physics (ERA5 Wind)   | MAE: {mae_era5:.4f} | RMSE: {rmse_era5:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# รวบรวมผลลัพธ์ทั้งหมด
final_results = pd.DataFrame([
    {"Model": "Persistence", "MAE": 0.1243, "RMSE": 0.1919},
    {"Model": "E1: SST Baseline", "MAE": 0.1205, "RMSE": 0.1836},
    {"Model": "Multi-Task (SST+SLP)", "MAE": 0.1203, "RMSE": 0.1828},
    {"Model": "Physics (NCEP)", "MAE": 0.1195, "RMSE": 0.1811},
    {"Model": "Physics (ERA5)", "MAE": 0.1213, "RMSE": 0.1822},
])

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(final_results))
width = 0.35

bars1 = ax.bar(x - width/2, final_results["MAE"], width, label='MAE (°C)', color='#4C72B0')
bars2 = ax.bar(x + width/2, final_results["RMSE"], width, label='RMSE (°C)', color='#DD8452')

# ไฮไลท์แท่งที่ชนะ (Physics NCEP)
ax.patches[6].set_facecolor('#55A868') # MAE ของ Physics NCEP
ax.patches[7].set_facecolor('#55A868') # RMSE ของ Physics NCEP

ax.set_ylabel('Error (°C)')
ax.set_title('Final Model Comparison: The Impact of Physics and Data Resolution')
ax.set_xticks(x)
ax.set_xticklabels(final_results["Model"], rotation=15, ha='right')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.4f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig("reports/final_comparison_v3.png", dpi=150)
plt.show()
print("✅ Saved: reports/final_comparison_v3.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import copy
import warnings

# Ignore warnings for cleaner output
warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# 1. Define Model and Dataset
# ---------------------------------------------------------
class PhysicsCNN(nn.Module):
    def __init__(self, in_channels, hidden=(32, 32, 16)):
        super().__init__()
        layers, prev = [], in_channels
        for h in hidden:
            layers += [nn.Conv2d(prev, h, 3, padding=1), nn.ReLU()]
            prev = h
        head = nn.Conv2d(prev, 1, 1)
        nn.init.zeros_(head.weight)
        nn.init.zeros_(head.bias)
        layers.append(head)
        self.net = nn.Sequential(*layers)
    
    def forward(self, x): 
        return self.net(x)

class PhysicsDataset(Dataset):
    def __init__(self, indices, features):
        self.indices = np.asarray(indices, dtype=int)
        self.features = features
    
    def __len__(self): 
        return len(self.indices)
    
    def __getitem__(self, i):
        t = self.indices[i]
        window = self.features[t - input_days:t]
        x = window.transpose(0, 3, 1, 2).reshape(-1, H, W)
        valid = np.isfinite(sst_model[t]) & np.isfinite(sst_model[t - 1]) & ocean_mask
        y = np.where(valid, sst_norm[t] - sst_norm[t - 1], 0.0).astype(np.float32)
        w = valid.astype(np.float32)
        return (
            torch.from_numpy(np.ascontiguousarray(x)), 
            torch.from_numpy(np.expand_dims(y, 0)), 
            torch.from_numpy(np.expand_dims(w, 0))
        )

# ---------------------------------------------------------
# 2. Prepare Data and Train Physics Model (NCEP)
# ---------------------------------------------------------
print("Preparing data...")
inputs_phys = ["sst", "wind"]
features_phys = np.concatenate([data_dict[k] for k in inputs_phys], axis=-1)
in_channels_phys = features_phys.shape[-1] * input_days

train_ds_phys = PhysicsDataset(train_targets, features_phys)
val_ds_phys = PhysicsDataset(val_targets, features_phys)
test_ds_phys = PhysicsDataset(test_targets, features_phys)

train_loader = DataLoader(train_ds_phys, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds_phys, batch_size=4, shuffle=False)

def physics_loss(pred_delta_sst, u_wind, v_wind):
    dT_dy, dT_dx = torch.gradient(pred_delta_sst[:, 0], dim=(1, 2))
    advection = u_wind[:, 0] * dT_dx + v_wind[:, 0] * dT_dy
    return torch.mean(advection ** 2)

# Train model
print("Training Physics-Informed Model (NCEP)...")
torch.manual_seed(42)
phys_model_ncep = PhysicsCNN(in_channels_phys).to(DEVICE)
optimizer = optim.Adam(phys_model_ncep.parameters(), lr=1e-3)
criterion = nn.MSELoss(reduction='none')

LAMBDA_PHYSICS = 0.01
EPOCHS_PHYS = 10  # Reduced for speed, can increase if needed
best_val_loss = float('inf')
best_state = copy.deepcopy(phys_model_ncep.state_dict())
patience_counter = 0

for epoch in range(EPOCHS_PHYS):
    phys_model_ncep.train()
    for bx, by, bw in train_loader:
        bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
        u_last = bx[:, -2, :, :].unsqueeze(1)
        v_last = bx[:, -1, :, :].unsqueeze(1)
        
        preds = phys_model_ncep(bx)
        data_loss = (((preds - by) ** 2) * bw).mean()
        p_loss = physics_loss(preds, u_last, v_last)
        total_loss = data_loss + LAMBDA_PHYSICS * p_loss
        
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

    # Validation
    phys_model_ncep.eval()
    val_loss = 0.0
    with torch.no_grad():
        for bx, by, bw in val_loader:
            bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
            preds = phys_model_ncep(bx)
            val_loss += (((preds - by) ** 2) * bw).mean().item() * bx.size(0)
    val_loss /= len(val_loader.dataset)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(phys_model_ncep.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 3:
            print(f"Early stopping at epoch {epoch+1}")
            break

# Load best weights and save to file
phys_model_ncep.load_state_dict(best_state)
torch.save(best_state, "phys_model.pth")
print("✅ Model trained and saved as 'phys_model.pth'")

# ---------------------------------------------------------
# 3. Generate Predictions
# ---------------------------------------------------------
phys_model_ncep.eval()
predictions = []
with torch.no_grad():
    for bx, _, _ in DataLoader(test_ds_phys, batch_size=4, shuffle=False):
        preds = phys_model_ncep(bx.to(DEVICE))
        predictions.append(preds.cpu().numpy()[:, 0])

delta_pred = np.concatenate(predictions, axis=0)
pred_sst_physics = (sst_norm[test_targets - 1] + delta_pred) * sst_std + sst_mean

# ---------------------------------------------------------
# 4. Regional Analysis (Warm Pool, Mid, Cold)
# ---------------------------------------------------------
print("\nAnalyzing regional performance...")

# Compute mean SST for 2021 (ignoring NaN)
mean_sst_2021 = np.nanmean(sst_model[split_idx:], axis=0)

# Create masks for each zone based on mean SST
warm_pool_mask = (mean_sst_2021 > 28.0) & ocean_mask
mid_zone_mask = ((mean_sst_2021 >= 25.0) & (mean_sst_2021 <= 28.0)) & ocean_mask
cold_zone_mask = (mean_sst_2021 < 25.0) & ocean_mask

print(f"Grid cells in Warm Pool (>28°C): {warm_pool_mask.sum()}")
print(f"Grid cells in Mid Zone (25-28°C): {mid_zone_mask.sum()}")
print(f"Grid cells in Cold Zone (<25°C): {cold_zone_mask.sum()}")

def evaluate_by_zone(prediction, actual, eval_mask):
    results = {}
    for zone_name, zone_mask in [
        ("Warm Pool", warm_pool_mask), 
        ("Mid Zone", mid_zone_mask), 
        ("Cold Zone", cold_zone_mask)
    ]:
        mask = eval_mask & zone_mask[None, :, :]
        if mask.sum() > 0:
            err = prediction[mask] - actual[mask]
            results[zone_name] = {
                "MAE": float(np.mean(np.abs(err))),
                "RMSE": float(np.sqrt(np.mean(err ** 2)))
            }
    return results

actual_test = sst_model[test_targets]
eval_mask = np.isfinite(actual_test) & np.isfinite(sst_model[test_targets - 1]) & ocean_mask[None]

physics_results = evaluate_by_zone(pred_sst_physics, actual_test, eval_mask)

print("\n=== Regional Performance: Physics-Informed Model (NCEP) ===")
for zone, metrics in physics_results.items():
    print(f"{zone:10s}: MAE = {metrics['MAE']:.4f} °C | RMSE = {metrics['RMSE']:.4f} °C")

# ---------------------------------------------------------
# 5. Plot Zones
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
zone_map = np.full_like(mean_sst_2021, np.nan)
zone_map[cold_zone_mask] = 1
zone_map[mid_zone_mask] = 2
zone_map[warm_pool_mask] = 3

cmap = plt.colormaps['RdYlBu_r'].resampled(3)
im = ax.pcolormesh(target_lon, target_lat, zone_map, cmap=cmap, shading="auto")
ax.set_title("Spatial Zones: Warm Pool vs Mid vs Cold Zone")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
cbar = fig.colorbar(im, ax=ax, ticks=[1.33, 2, 2.66])
cbar.ax.set_yticklabels(['Cold Zone', 'Mid Zone', 'Warm Pool'])
plt.tight_layout()
plt.savefig("reports/regional_zones.png", dpi=150)
plt.show()
print("✅ Saved: reports/regional_zones.png")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Create a DataFrame for plotting
regional_df = pd.DataFrame([
    {"Zone": "Warm Pool", "MAE": 0.1081, "RMSE": 0.1571},
    {"Zone": "Mid Zone", "MAE": 0.1427, "RMSE": 0.2205},
    {"Zone": "Cold Zone", "MAE": 0.1609, "RMSE": 0.2471},
])

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(regional_df))
width = 0.35

bars1 = ax.bar(x - width/2, regional_df["MAE"], width, label='MAE (°C)', color='#4C72B0')
bars2 = ax.bar(x + width/2, regional_df["RMSE"], width, label='RMSE (°C)', color='#DD8452')

ax.set_ylabel('Error (°C)')
ax.set_title('Regional Performance: Warm Pool vs Mid vs Cold Zone')
ax.set_xticks(x)
ax.set_xticklabels(regional_df["Zone"])
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.4f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig("reports/regional_performance.png", dpi=150)
plt.show()
print("✅ Saved: reports/regional_performance.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------------------------------------------------
# 1. Calculate Wind Speed and SST Change
# ---------------------------------------------------------
# Compute wind speed magnitude from normalized U and V
u_wind = data_dict["wind"][..., 0]
v_wind = data_dict["wind"][..., 1]
wind_speed = np.sqrt(u_wind**2 + v_wind**2)

# Compute absolute SST change per day (in normalized units first)
sst_change = np.abs(sst_norm[1:] - sst_norm[:-1])
wind_speed_aligned = wind_speed[:-1] # Align with SST change

# ---------------------------------------------------------
# 2. Analyze Correlation per Zone
# ---------------------------------------------------------
def analyze_wind_impact(zone_mask, zone_name):
    # Extract values only for the specific zone
    zone_wind = wind_speed_aligned[:, zone_mask].mean(axis=1)
    zone_sst_change = sst_change[:, zone_mask].mean(axis=1)
    
    # Calculate correlation
    corr, p_value = pearsonr(zone_wind, zone_sst_change)
    print(f"{zone_name:10s} | Correlation (Wind vs SST Change): {corr:.4f} (p-value: {p_value:.2e})")
    return corr

print("=== Wind Forcing Analysis ===")
corr_warm = analyze_wind_impact(warm_pool_mask, "Warm Pool")
corr_mid = analyze_wind_impact(mid_zone_mask, "Mid Zone")
corr_cold = analyze_wind_impact(cold_zone_mask, "Cold Zone")

# ---------------------------------------------------------
# 3. Plot the Relationship
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
zones = [(warm_pool_mask, "Warm Pool", corr_warm), 
         (mid_zone_mask, "Mid Zone", corr_mid), 
         (cold_zone_mask, "Cold Zone", corr_cold)]

for ax, (mask, name, corr) in zip(axes, zones):
    zone_wind = wind_speed_aligned[:, mask].mean(axis=1)
    zone_sst_change = sst_change[:, mask].mean(axis=1)
    
    ax.scatter(zone_wind, zone_sst_change, alpha=0.3, s=10, color='blue')
    ax.set_title(f"{name} (r = {corr:.3f})")
    ax.set_xlabel("Mean Wind Speed (Normalized)")
    ax.set_ylabel("Mean SST Change (Normalized)")
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig("reports/wind_forcing_analysis.png", dpi=150)
plt.show()
print("✅ Saved: reports/wind_forcing_analysis.png")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import copy
import warnings

warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# 1. Define Helper Classes and Functions
# ---------------------------------------------------------
class ForecastDataset(Dataset):
    def __init__(self, indices, features):
        self.indices = np.asarray(indices, dtype=int)
        self.features = features
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        t = self.indices[i]
        window = self.features[t - input_days:t]
        x = window.transpose(0, 3, 1, 2).reshape(-1, H, W)
        valid = np.isfinite(sst_model[t]) & np.isfinite(sst_model[t - 1]) & ocean_mask
        y = np.where(valid, sst_norm[t] - sst_norm[t - 1], 0.0).astype(np.float32)
        w = valid.astype(np.float32)
        return (torch.from_numpy(np.ascontiguousarray(x)), 
                torch.from_numpy(np.expand_dims(y, 0)), 
                torch.from_numpy(np.expand_dims(w, 0)))

class SimpleCNN(nn.Module):
    def __init__(self, in_channels, hidden=(16, 16, 8)):
        super().__init__()
        layers, prev = [], in_channels
        for h in hidden:
            layers += [nn.Conv2d(prev, h, 3, padding=1), nn.ReLU()]
            prev = h
        head = nn.Conv2d(prev, 1, 1)
        nn.init.zeros_(head.weight); nn.init.zeros_(head.bias)
        layers.append(head)
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def train_simple_model(inputs_list, name, epochs=5):
    features = np.concatenate([data_dict[k] for k in inputs_list], axis=-1)
    in_channels = features.shape[-1] * input_days
    train_ds = ForecastDataset(train_targets, features)
    val_ds = ForecastDataset(val_targets, features)
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)
    
    model = SimpleCNN(in_channels).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss(reduction='none')
    
    print(f"Training {name}...")
    for epoch in range(epochs):
        model.train()
        for bx, by, bw in train_loader:
            bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
            optimizer.zero_grad()
            preds = model(bx)
            loss = (((preds - by) ** 2) * bw).mean()
            loss.backward()
            optimizer.step()
    return model

# ---------------------------------------------------------
# 2. Generate Predictions for All Models
# ---------------------------------------------------------
test_targets = np.arange(split_idx, T)
actual_test = sst_model[test_targets]
eval_mask = np.isfinite(actual_test) & np.isfinite(sst_model[test_targets - 1]) & ocean_mask[None]

# --- A. Baseline CNN (SST only) ---
try:
    # Try to load if exists, else train
    model_base = train_simple_model(["sst"], "Baseline (SST only)", epochs=3)
    features_base = data_dict["sst"]
    test_ds_base = ForecastDataset(test_targets, features_base)
    preds_list = []
    model_base.eval()
    with torch.no_grad():
        for bx, _, _ in DataLoader(test_ds_base, batch_size=4):
            preds_list.append(model_base(bx.to(DEVICE)).cpu().numpy()[:, 0])
    pred_sst_base = (sst_norm[test_targets - 1] + np.concatenate(preds_list)) * sst_std + sst_mean
    print("✅ Baseline Prediction Ready")
except Exception as e:
    print(f"Error Baseline: {e}")

# --- B. Multi-Task CNN (SST + SLP) ---
# To save time, we'll just use a simple CNN with SST+SLP as a proxy for ensemble diversity
model_multi = train_simple_model(["sst", "slp"], "Multi-Var (SST + SLP)", epochs=3)
features_multi = np.concatenate([data_dict["sst"], data_dict["slp"]], axis=-1)
test_ds_multi = ForecastDataset(test_targets, features_multi)
preds_list = []
model_multi.eval()
with torch.no_grad():
    for bx, _, _ in DataLoader(test_ds_multi, batch_size=4):
        preds_list.append(model_multi(bx.to(DEVICE)).cpu().numpy()[:, 0])
pred_sst_multi = (sst_norm[test_targets - 1] + np.concatenate(preds_list)) * sst_std + sst_mean
print("✅ Multi-Var Prediction Ready")

# --- C. Physics CNN (SST + Wind) ---
# Load the saved Physics model
class PhysicsCNN(nn.Module):
    def __init__(self, in_channels, hidden=(32, 32, 16)):
        super().__init__()
        layers, prev = [], in_channels
        for h in hidden:
            layers += [nn.Conv2d(prev, h, 3, padding=1), nn.ReLU()]
            prev = h
        head = nn.Conv2d(prev, 1, 1)
        layers.append(head)
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

features_phys = np.concatenate([data_dict["sst"], data_dict["wind"]], axis=-1)
test_ds_phys = ForecastDataset(test_targets, features_phys)
phys_model = PhysicsCNN(features_phys.shape[-1] * input_days).to(DEVICE)
try:
    phys_model.load_state_dict(torch.load("phys_model.pth", map_location=DEVICE))
    print("✅ Loaded Physics Model from file")
except:
    print("⚠️ Physics Model not found, training quickly...")
    phys_model = train_simple_model(["sst", "wind"], "Physics Proxy", epochs=3)

preds_list = []
phys_model.eval()
with torch.no_grad():
    for bx, _, _ in DataLoader(test_ds_phys, batch_size=4):
        preds_list.append(phys_model(bx.to(DEVICE)).cpu().numpy()[:, 0])
pred_sst_physics = (sst_norm[test_targets - 1] + np.concatenate(preds_list)) * sst_std + sst_mean
print("✅ Physics Prediction Ready")

# ---------------------------------------------------------
# 3. Ensemble Forecasting
# ---------------------------------------------------------
print("\nCombining predictions into Ensemble...")
pred_sst_ensemble = (pred_sst_base + pred_sst_multi + pred_sst_physics) / 3.0

def calculate_metrics(actual, predicted, mask):
    err = predicted[mask] - actual[mask]
    return float(np.mean(np.abs(err))), float(np.sqrt(np.mean(err ** 2)))

base_mae, base_rmse = calculate_metrics(actual_test, pred_sst_base, eval_mask)
multi_mae, multi_rmse = calculate_metrics(actual_test, pred_sst_multi, eval_mask)
phys_mae, phys_rmse = calculate_metrics(actual_test, pred_sst_physics, eval_mask)
ens_mae, ens_rmse = calculate_metrics(actual_test, pred_sst_ensemble, eval_mask)

results = pd.DataFrame([
    {"Model": "Baseline CNN", "MAE": base_mae, "RMSE": base_rmse},
    {"Model": "Multi-Var CNN", "MAE": multi_mae, "RMSE": multi_rmse},
    {"Model": "Physics CNN", "MAE": phys_mae, "RMSE": phys_rmse},
    {"Model": "Ensemble (Avg)", "MAE": ens_mae, "RMSE": ens_rmse}
])

print("\n=== Ensemble Forecasting Results ===")
display(results.round(4))

# Plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results))
width = 0.35
bars1 = ax.bar(x - width/2, results["MAE"], width, label='MAE (°C)', color='#4C72B0')
bars2 = ax.bar(x + width/2, results["RMSE"], width, label='RMSE (°C)', color='#DD8452')
ax.set_ylabel('Error (°C)')
ax.set_title('Model Comparison: Individual vs Ensemble')
ax.set_xticks(x)
ax.set_xticklabels(results["Model"], rotation=15, ha='right')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.4f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig("reports/ensemble_comparison.png", dpi=150)
plt.show()
print("✅ Saved: reports/ensemble_comparison.png")

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.models.video import swin3d_t
import numpy as np
import gc

# ---------------------------------------------------------
# 1. Setup Device
# ---------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {DEVICE}")

gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------------------
# 2. Define Swin Transformer (Fixed Head for 1D Output)
# ---------------------------------------------------------
class SwinSSTForecaster(nn.Module):
    def __init__(self, out_features=1):
        super().__init__()
        self.swin = swin3d_t(weights=None)
        
        # Swin3D outputs a 1D vector of size 768 (for Swin-T)
        in_features = self.swin.head.in_features
        # Replace with a Linear layer that predicts the single Delta value
        self.swin.head = nn.Linear(in_features, out_features)

    def forward(self, x):
        # x: (B, 7, H, W) -> (B, 1, 7, H, W)
        B, C, H, W = x.shape
        x = x.reshape(B, INPUT_DAYS, 1, H, W).permute(0, 2, 1, 3, 4) 
        
        # Duplicate channel to 3 (RGB trick)
        x = x.repeat(1, 3, 1, 1, 1) 
        
        # Forward pass returns (B, 1) because we changed head to Linear
        return self.swin(x)

# ---------------------------------------------------------
# 3. Prepare Data (SST Only - Modified for Vector Output)
# ---------------------------------------------------------
class TransformerDataset(Dataset):
    def __init__(self, indices, features):
        self.indices = np.asarray(indices, dtype=int)
        self.features = features
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        t = self.indices[i]
        window = self.features[t - INPUT_DAYS:t]
        x = window[..., 0].astype(np.float32)
        valid = np.isfinite(sst_model[t]) & np.isfinite(sst_model[t - 1]) & ocean_mask
        
        # 🔧 Instead of full map, we take the MEAN of the SST change as a target
        # (Since Swin outputs a single value, we predict the mean change)
        target_delta = np.where(valid, sst_norm[t] - sst_norm[t - 1], 0.0).mean()
        
        return (torch.from_numpy(x), 
                torch.tensor([target_delta], dtype=torch.float32))

features_sst = data_dict["sst"]
train_ds = TransformerDataset(train_targets, features_sst)
val_ds = TransformerDataset(val_targets, features_sst)
test_ds = TransformerDataset(test_targets, features_sst)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)

# ---------------------------------------------------------
# 4. Train with Mixed Precision (AMP)
# ---------------------------------------------------------
model = SwinSSTForecaster().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=5e-5) 
criterion = nn.MSELoss() # MSE for 1D vector
scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else torch.amp.GradScaler('cpu')

EPOCHS = 5 
PATIENCE = 2
best_val_loss = float('inf')
wait = 0

print("\n🚀 Training Swin Transformer (Vector Output Mode)...")
for epoch in range(EPOCHS):
    model.train()
    for bx, by in train_loader:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16):
            preds = model(bx)
            loss = criterion(preds, by)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16):
                preds = model(bx)
                val_loss += criterion(preds, by).item() * bx.size(0)
    val_loss /= len(val_loader.dataset)
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Val Loss: {val_loss:.6f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_transformer.pth")
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"⏹️ Early stopping at epoch {epoch+1}")
            break

print("✅ Training Complete!")

# ---------------------------------------------------------
# 5. Evaluate (Calculate MAE for Mean Delta)
# ---------------------------------------------------------
model.load_state_dict(torch.load("best_transformer.pth"))
model.eval()
predictions = []
targets = []
with torch.no_grad():
    for bx, by in DataLoader(test_ds, batch_size=1, shuffle=False):
        preds = model(bx.to(DEVICE))
        predictions.append(preds.cpu().numpy())
        targets.append(by.numpy())

pred_delta = np.concatenate(predictions, axis=0)
actual_delta = np.concatenate(targets, axis=0)

# Calculate Mean Absolute Error for the scalar prediction
mae = np.mean(np.abs(pred_delta - actual_delta))
print(f"\n✅ Swin Transformer (Mean Delta) | MAE: {mae:.4f}")

🚀 Using device: cuda

🚀 Training Swin Transformer (Vector Output Mode)...
Epoch 01/5 | Val Loss: 0.041351
Epoch 02/5 | Val Loss: 0.000152
Epoch 03/5 | Val Loss: 0.000867
Epoch 04/5 | Val Loss: 0.000473
⏹️ Early stopping at epoch 4
✅ Training Complete!

✅ Swin Transformer (Mean Delta) | MAE: 0.0081
